# Module 04: Tokenization


# 4.1 Tokenization Basics


## ✂️ How Tokenization Works in spaCy

Tokenization is the task of splitting text into meaningful segments, called tokens. These can be words, punctuation marks, numbers, etc.

Unlike most other pipeline components, the Tokenizer in spaCy does **not** use a statistical model. It uses **language-specific rules**.

### The Tokenization Algorithm
1. Iterate over space-separated substrings.
2. Check if the substring matches an exception rule (e.g., "don't" -> "do" and "n't").
3. If no exception matches, check for prefixes, suffixes, and infixes (like punctuation).
4. Split the substring accordingly and repeat.


In [2]:
import spacy

nlp = spacy.load("en_core_web_sm")
text = "Let's go to N.Y.C.!"
doc = nlp(text)

print("Tokens:")
for i, token in enumerate(doc):
    print(f"[{i}] {token.text}")


Tokens:
[0] Let
[1] 's
[2] go
[3] to
[4] N.Y.C.
[5] !


## ⚠️ Edge Cases

Notice in the example above:
- `Let's` was split into `Let` and `'s`.
- `N.Y.C.` was kept together as a single token, despite having punctuation, because it's a known abbreviation.
- `!` was separated from `N.Y.C.`.

This behavior is highly language-dependent. The English tokenizer knows English rules, but the German tokenizer would behave differently.


In [3]:
# Compare English vs. a blank English tokenizer vs. raw splitting
text2 = "I've paid $15.50 for this!"

# Python split
print("Python split:", text2.split())

# spaCy tokenization
doc2 = nlp(text2)
print("\nspaCy split:", [t.text for t in doc2])


Python split: ["I've", 'paid', '$15.50', 'for', 'this!']

spaCy split: ['I', "'ve", 'paid', '$', '15.50', 'for', 'this', '!']


Notice how spaCy correctly isolates the `$` and the `!` without breaking the float `15.50`.



<br><br>

---

<br><br>


# 4.2 Customizing the Tokenizer


## 🛠️ Adding Special Cases

Sometimes you have domain-specific terms that spaCy splits incorrectly. For example, maybe in your company, "gimme" shouldn't be split into "gim" and "me", or maybe a specific serial number format should be kept as one token.

You can add **special cases** directly to the tokenizer.


In [ ]:
import spacy
from spacy.symbols import ORTH

nlp = spacy.load("en_core_web_sm")

# By default, spaCy splits "don't" into "do" and "n't"
text = "I really don't know."
print("Before custom rule:", [t.text for t in nlp(text)])

# Let's say we want to treat "don't" as a single token instead
# We tell the tokenizer that whenever it sees "don't", it should map to a single token
special_case = [{ORTH: "don't"}]
nlp.tokenizer.add_special_case("don't", special_case)

print("After custom rule:", [t.text for t in nlp(text)])


## 🔄 Modifying Prefixes, Suffixes, and Infixes

If you want to change how the tokenizer handles punctuation globally, you can modify the regex patterns it uses for prefixes, suffixes, and infixes.

For example, let's say you want to split tokens on hyphens `-` (which spaCy normally keeps together in words like "state-of-the-art").


In [ ]:
import re

# Get current infix rules
# (Infixes are characters that split tokens from the middle)
infixes = nlp.Defaults.infixes

# We add a hyphen to the regex rules
custom_infixes = tuple(list(infixes) + [r"-"])

# Compile the new regex
infix_re = spacy.util.compile_infix_regex(custom_infixes)

# Update the tokenizer
nlp.tokenizer.infix_finditer = infix_re.finditer

text = "state-of-the-art technology"
print("Custom hyphen splitting:", [t.text for t in nlp(text)])


## 💡 When to Customize?
- Use **Special Cases** for specific words, acronyms, or IDs.
- Use **Prefix/Suffix/Infix rules** to change global punctuation behavior.
- Use **Retokenization (Merging)** (from Module 2) to merge tokens *after* they've been processed if you want to keep the statistical model's original accuracy.



<br><br>

---

<br><br>


# 4.3 Sentence Segmentation


## 📝 Finding Sentence Boundaries

By default, spaCy uses the Dependency Parser to figure out where sentences begin and end. This is highly accurate because it understands the grammar of the sentence, not just looking for periods.

You access sentences via `doc.sents` (which returns a generator of `Span` objects).


In [1]:
import spacy
nlp = spacy.load("en_core_web_sm")

text = "Here's a sentence. Dr. Smith is here! He brought apples, oranges, etc. And we are happy."
doc = nlp(text)

print("Default Dependency Parser Segmentation:")
for i, sent in enumerate(doc.sents):
    print(f"[{i+1}] {sent.text}")


Default Dependency Parser Segmentation:
[1] Here's a sentence.
[2] Dr. Smith is here!
[3] He brought apples, oranges, etc.
[4] And we are happy.


## ⚡ The Sentencizer (Fast, Rule-Based)

If you are only doing text extraction and don't care about dependencies, running the full parser is too slow.
Instead, you can use the **`sentencizer`** pipeline component. It uses simple punctuation rules (like splitting on `.` or `!`).


In [2]:
nlp_fast = spacy.blank("en") # Create an empty pipeline
nlp_fast.add_pipe("sentencizer") # Add only the sentencizer

doc_fast = nlp_fast(text)
print("Rule-based Sentencizer Segmentation:")
for i, sent in enumerate(doc_fast.sents):
    print(f"[{i+1}] {sent.text}")


Rule-based Sentencizer Segmentation:
[1] Here's a sentence.
[2] Dr. Smith is here!
[3] He brought apples, oranges, etc.
[4] And we are happy.


Notice that the `sentencizer` might make mistakes with things like "Dr. Smith" or "etc." if not configured properly, whereas the Dependency Parser is much smarter about them!


## 🛠️ Custom Sentence Boundaries

What if your text is split by newlines `\n` instead of periods? You can write a custom component to force sentence boundaries before the parser runs by modifying `token.is_sent_start`.


In [3]:
from spacy.language import Language

@Language.component("custom_newline_boundaries")
def set_custom_boundaries(doc):
    for token in doc[:-1]:
        if token.text == "\n":
            doc[token.i + 1].is_sent_start = True
    return doc

# Add our custom component before the parser
nlp = spacy.load("en_core_web_sm")
nlp.add_pipe("custom_newline_boundaries", before="parser")

text = "First line\nSecond line without period\nThird line"
doc = nlp(text)

print("Custom Boundaries:")
for sent in doc.sents:
    print(f"- {sent.text.strip()}")


Custom Boundaries:
- First line
- Second line without period
- Third line



<br><br>

---

<br><br>


# 4.4 Text Normalization


## 🧹 Cleaning Text for Machine Learning

Before feeding text into ML algorithms or search indices, it is common to "normalize" it. Normalization usually involves:
1. Lowercasing
2. Removing punctuation
3. Removing stop words (very common words like 'the', 'is', 'and')
4. Using Lemmas instead of exact words

spaCy makes this incredibly easy because the tokens already have properties for all of these!


In [1]:
import spacy
nlp = spacy.load("en_core_web_sm")

text = "The QUICK brown foxes jumped over the lazy dogs!!! 😊"
doc = nlp(text)


### 1. Simple Filtering
Let's remove punctuation, spaces, and stop words, and get the lowercase string.


In [2]:
cleaned_tokens = []

for token in doc:
    # Skip punctuation and spaces
    if token.is_punct or token.is_space:
        continue
    # Skip stop words
    if token.is_stop:
        continue
        
    # Append the lowercase text
    cleaned_tokens.append(token.lower_)

print(f"Original: {text}")
print(f"Cleaned:  {' '.join(cleaned_tokens)}")


Original: The QUICK brown foxes jumped over the lazy dogs!!! 😊
Cleaned:  quick brown foxes jumped lazy dogs 😊


### 2. Lemmatization (Base forms)
If you want to map "foxes" to "fox" and "jumped" to "jump", use `token.lemma_` instead of `token.lower_`.


In [3]:
lemma_tokens = []

for token in doc:
    if not token.is_punct and not token.is_space and not token.is_stop:
        lemma_tokens.append(token.lemma_.lower())

print(f"Lemmatized: {' '.join(lemma_tokens)}")


Lemmatized: quick brown fox jump lazy dog 😊


## 🛑 Modifying the Stop Word List

spaCy provides a default set of stop words for every language. You can view, add, or remove words from this list.


In [4]:
# View the stop words (first 10)
print("Some default stop words:", list(nlp.Defaults.stop_words)[:10])

# Let's say we want to add 'fox' as a stop word
nlp.Defaults.stop_words.add("fox")
nlp.vocab["fox"].is_stop = True  # We also must update the vocab!

# Let's check
doc = nlp("The fox jumped.")
print("\nIs 'fox' a stop word now?", doc[1].is_stop)


Some default stop words: ['therein', 'hereafter', 'none', 'not', 'further', 'only', 'anyhow', 'us', 'her', 'yet']

Is 'fox' a stop word now? True



<br><br>

---

<br><br>


# 4.5 Handling URLs, Emails, and Special Text


## 📧 Built-in Detectors

Web text is messy. It contains URLs, emails, hashtags, and emojis. spaCy's tokenizer is built to handle modern web text right out of the box.

Tokens come with convenient `like_url` and `like_email` flags!


In [1]:
import spacy
nlp = spacy.load("en_core_web_sm")

text = "Contact support at help@spacy.io or visit https://spacy.io/ for docs! Follow #NLP 🎉"
doc = nlp(text)

print("--- SPECIAL TOKENS ---")
for token in doc:
    if token.like_email:
        print(f"Email found: {token.text}")
    elif token.like_url:
        print(f"URL found: {token.text}")


--- SPECIAL TOKENS ---
Email found: help@spacy.io
URL found: https://spacy.io/


## 📱 Hashtags and Mentions

spaCy doesn't have a built-in `like_hashtag` property, but it's very easy to create one using the Matcher (which we will cover extensively in Module 8), or by checking if the first character is `#`.

However, by default, spaCy might split hashtags if they have punctuation. Let's see.


In [2]:
print("Checking tokens for hashtags:")
for token in doc:
    if token.text.startswith("#"):
        print(f"Potential Hashtag: {token.text}")


Checking tokens for hashtags:
Potential Hashtag: #


## 😎 Handling Emojis

Emojis are treated as individual tokens, which is great for sentiment analysis or filtering.


In [3]:
# Let's extract all emojis using the `is_punct` and `is_alpha` properties
# (Emojis are usually neither, but the safest way is using unicode or a library)
# However, spaCy keeps them as clean tokens.

emojis = [token.text for token in doc if not token.is_alpha and not token.is_punct and not token.is_space]
print(f"Emojis extracted: {emojis}")


Emojis extracted: ['help@spacy.io', 'https://spacy.io/', '🎉']


## 🎉 Summary of Module 4

You have mastered how text is broken down into its fundamental units!
- You learned how the Tokenizer uses rules, and how to customize those rules with Special Cases and Infixes.
- You learned the difference between the robust Dependency Parser and the blazing fast Sentencizer for sentence splitting.
- You know how to filter, lowercase, and lemmatize text to prepare it for machine learning.
- You learned how to easily extract URLs and Emails.

In **Module 5: Linguistic Annotations**, we will dive deeper into Part-of-Speech tagging, dependency trees, and visualizing the syntax of your text.
